# 02 - Data Preparation

Walks through cleaning, encoding, stratified split, and **SMOTE oversampling on the training partition only**. Calls the same `preprocess()` function used in the pipeline.

In [5]:
import sys
from pathlib import Path

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
sys.path.insert(0, str(PROJECT_ROOT / 'code'))

import pandas as pd
from utils.paths import DATA_PROCESSED, MODELS, TARGET_COL
from data_preparation import preprocess

## Run preprocessing

This will:
1. Drop duplicates and impute any missing values.
2. One-hot encode categorical features (persist the encoder).
3. Stratified 80/20 train/test split.
4. Apply SMOTE on the training partition only.
5. Write CSVs to `data/processed/` and artifacts to `outputs/models/`.

In [6]:
result = preprocess.preprocess()
X_train, X_test, y_train, y_test = result['X_train'], result['X_test'], result['y_train'], result['y_test']
print('X_train:', X_train.shape, ' X_test:', X_test.shape)

2026-05-26 15:52:15 | INFO     | preprocess | Loading raw data from C:\Users\U1\Documents\fraud-1\data\raw\synthetic_fraud_dataset.csv
2026-05-26 15:52:15 | INFO     | preprocess | Raw shape: (10000, 10)
2026-05-26 15:52:15 | INFO     | preprocess | Post-deduplication shape: (10000, 10)
2026-05-26 15:52:15 | INFO     | preprocess | Numeric features (4): ['amount', 'hour', 'device_risk_score', 'ip_risk_score']
2026-05-26 15:52:15 | INFO     | preprocess | Categorical features (3): ['transaction_type', 'merchant_category', 'country']
2026-05-26 15:52:15 | INFO     | preprocess | Engineered feature matrix shape: (10000, 19)
2026-05-26 15:52:15 | INFO     | preprocess | Pre-SMOTE | train=(8000, 19) test=(2000, 19) | train positives=400 (5.00%) | test positives=100 (5.00%)
2026-05-26 15:52:15 | INFO     | preprocess | Post-SMOTE | train=(15200, 19) | positives=7600 (50.00%)
2026-05-26 15:52:15 | INFO     | preprocess | Post-SMOTE data types | X_train_res: DataFrame, y_train_res: Series
2026

## Class balance before vs after SMOTE

In [7]:
post = pd.Series(y_train).value_counts().rename('post_smote_train')
holdout = pd.Series(y_test).value_counts().rename('holdout_test')
comparison = pd.concat([post, holdout], axis=1).fillna(0).astype(int)
comparison['post_smote_train_pct'] = (comparison['post_smote_train'] / comparison['post_smote_train'].sum() * 100).round(2)
comparison['holdout_test_pct'] = (comparison['holdout_test'] / comparison['holdout_test'].sum() * 100).round(2)
comparison

,post_smote_train,holdout_test,post_smote_train_pct,holdout_test_pct
is_fraud,,,,
0,7600,1900,50.0,95.0
1,7600,100,50.0,5.0


Holdout intentionally keeps the original imbalance so reported precision/recall/F1 reflect production conditions.

In [8]:
import json
schema = json.loads((MODELS / 'feature_schema.json').read_text())
print('numeric:', schema['numeric'])
print('categorical:', schema['categorical'])
print('total features after encoding:', len(schema['feature_columns']))

numeric: ['amount', 'hour', 'device_risk_score', 'ip_risk_score']
categorical: ['transaction_type', 'merchant_category', 'country']
total features after encoding: 19
